# B1 – Local chatbot with pretrained models (CPU, Docker)

Runs each small model with the pipeline from the practice PDF (float32 for Intel CPUs), logs load time, latency and tokens/s, and sends flu-symptom prompts as a diagnosis request. Output: `results/b1_flu.csv` (paste/quote it in the report).

In [1]:
import sys, os, json, time, gc
sys.path.insert(0, "src")
from dotenv import load_dotenv; load_dotenv()
import pandas as pd
pd.set_option("display.max_colwidth", 200); pd.set_option("display.width", 200)
from vignettes import load_vignettes, sample_vignettes
from protocol import run_case, SYSTEM_DOCTOR
import clients, store, metrics

In [2]:
from pathlib import Path

# Move up from `notebook/` to project root once, even if re-run
if Path.cwd().name == "notebooks":
  os.chdir("..")

In [4]:
VIGNETTES = load_vignettes("data/Clinical_Vignettes_26_27.txt")
print(len(VIGNETTES), "clean 4-line vignettes")

117 clean 4-line vignettes


In [5]:
import torch, transformers
transformers.set_seed(0)
print(torch.__version__, "cuda:", torch.cuda.is_available(), "threads:", torch.get_num_threads())

MODEL_NAMES = [
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "microsoft/Phi-3.5-mini-instruct",      # 3.8B: slow on CPU, needs ~16 GB RAM in float32
]
SYSTEMS = {"generic": "You are a helpful assistant.", "physician": SYSTEM_DOCTOR}
FLU_PROMPTS = {
  "flu_brief": "I have had fever, body aches, a dry cough, a sore throat and tiredness for 3 days. What do I have?",
  "flu_detailed": ("I am a 46-year-old adult. Yesterday I suddenly developed fever (38.7 C), muscle and joint aches, "
                   "headache, dry cough, sore throat and extreme fatigue. Several coworkers are ill. "
                   "Please give me your most likely diagnosis and a short differential."),
}

2.15.0+cpu cuda: False threads: 12


In [6]:
rows = []
for name in MODEL_NAMES:
    t0 = time.time()
    bot = clients.make_local_hf(name, max_new_tokens=256, temperature=0.7)   # settings from the PDF
    load_s = time.time() - t0
    for sys_key, system in SYSTEMS.items():
        for p_key, prompt in FLU_PROMPTS.items():
            msgs = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
            r = bot(msgs)
            rows.append({"model": name, "system": sys_key, "prompt": p_key, "load_s": round(load_s, 1),
                         "latency_s": round(r["latency_s"], 1), "completion_tokens": r["completion_tokens"],
                         "tok_per_s": round(r["completion_tokens"] / r["latency_s"], 2), "answer": r["text"]})
            print(f"[{name} | {sys_key} | {p_key}] {r['latency_s']:.0f}s\n{r['text']}\n" + "-"*80)
    del bot; gc.collect()
pd.DataFrame(rows).to_csv("results/b1_flu.csv", index=False)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

... (40 outputs left out)

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

[microsoft/Phi-3.5-mini-instruct | generic | flu_brief] 213s
I'm not a doctor, but I can provide some general information. The symptoms you're describing—fever, body aches, dry cough, sore throat, and tiredness—can be associated with a variety of conditions, ranging from common viral infections to more serious illnesses. Here are a few possibilities, but it's crucial to consult with a healthcare professional for an accurate diagnosis and appropriate treatment:

1. **Viral Infection**: Viruses such as the common cold, influenza (flu), or COVID-19 (caused by the SARS-CoV-2 virus) can cause these symptoms. Influenza and COVID-19, in particular, have been widely circulating, and the symptoms overlap significantly.

2. **Bacterial Infection**: Bacterial infections like strep throat (caused by Streptococcus bacteria) can cause a sore throat, fever, and body aches.

3. **Other Infections**: Other viral infections like mononucleosis (caused by the Epstein-B
------------------------------------

## Behaviour summary (for the report)
Fill the qualitative columns yourself after reading the answers (names flu? hedges? hallucinates? ignores the physician prompt? recommends seeing a doctor?).

In [7]:
df = pd.read_csv("results/b1_flu.csv")
display(df.groupby("model")[["load_s", "latency_s", "tok_per_s"]].mean().round(2))
df["names_influenza"] = df["answer"].str.contains("influenza|flu", case=False)
display(df.groupby(["model", "system"])["names_influenza"].mean().unstack())

,load_s,latency_s,tok_per_s
model,,,
HuggingFaceTB/SmolLM2-1.7B-Instruct,138.8,101.15,1.12
Qwen/Qwen2.5-1.5B-Instruct,121.0,137.62,1.17
TinyLlama/TinyLlama-1.1B-Chat-v1.0,87.0,51.47,3.99
microsoft/Phi-3.5-mini-instruct,291.0,206.78,1.24


system,generic,physician
model,,
HuggingFaceTB/SmolLM2-1.7B-Instruct,1.0,1.0
Qwen/Qwen2.5-1.5B-Instruct,1.0,1.0
TinyLlama/TinyLlama-1.1B-Chat-v1.0,1.0,0.5
microsoft/Phi-3.5-mini-instruct,1.0,1.0
